In [ ]:
import pandas as pd
import time
from tqdm import tqdm
import numpy as np
import os
import logging
import time
import matplotlib.pyplot as plt
import re

import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.Chem import rdFingerprintGenerator
import pubchempy as pcp
from thermo.chemical import Chemical

In [ ]:
def get_properties_from_smiles(smiles):
    # Parse SMILES with RDKit
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception as e:
        print(f"Error parsing SMILES '{smiles}': {e}")
        mol = None
    
    # 1. Molar Mass
    try:
        molar_mass = Chem.Descriptors.MolWt(mol)
    except Exception as e:
        print(f"Error calculating molar mass for '{smiles}': {e}")
        molar_mass = None
    
    # 2. Net Charge
    try:
        charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception as e:
        print(f"Charge calculation failed for '{smiles}': {e}")
        charge = None
    
    # 2.5. Name from SMILES using PubChemPy
    try:
        compounds = pcp.get_compounds(smiles, 'smiles')
        name = compounds[0].iupac_name if compounds else None
    except Exception as e:
        print(f"PubChem name lookup failed: {e}")
        name = None
    
    # 3. Density (via Chemical)
    try:
        density = Chemical(name).rho / 1000  # Convert from kg/m^3 to g/cm^3
    except Exception as e:
        print(f"Density lookup failed: {e}")
        density = None
        
    return {
        'Molar_Mass_g_mol': molar_mass,
        'Charge': charge,
        'Density_g_cm3': density
    }

# Example usage
smiles = 'CCO'  # ethanol
properties = get_properties_from_smiles(smiles)

for key, value in properties.items():
    print(f"{key}: {value}")


# From SkinPerm_exp

In [ ]:
filename='SkinPerm_exp_250707'
df = pd.read_csv(f'../data/processed/{filename}_processed.csv')
df

In [ ]:
found_rows = df[df['SMILES_PubChem'] != 'Not found in PubChem']
found_rows

In [ ]:
found_rows.drop_duplicates(subset='can_SMILES', inplace=True)
found_rows.reset_index(drop=True, inplace=True)
found_rows

In [ ]:
found_rows['source'] = filename
found_rows

In [ ]:
found_rows.Compound_Name.nunique()

In [ ]:
found_rows.can_SMILES.nunique()

In [ ]:
def get_properties_from_smiles(smiles):
    # Parse SMILES with RDKit
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception as e:
        print(f"Error parsing SMILES '{smiles}': {e}")
        mol = None
    
    # 1. Molar Mass
    try:
        molar_mass = Chem.Descriptors.MolWt(mol)
    except Exception as e:
        print(f"Error calculating molar mass for '{smiles}': {e}")
        molar_mass = None
    
    # 2. Net Charge
    try:
        charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception as e:
        print(f"Charge calculation failed for '{smiles}': {e}")
        charge = None
    
    # 2.5. Name from SMILES using PubChemPy
    try:
        compounds = pcp.get_compounds(smiles, 'smiles')
        name = compounds[0].iupac_name if compounds else None
    except Exception as e:
        print(f"PubChem name lookup failed: {e}")
        name = None
    
    # 3. Density (via Chemical)
    try:
        density = Chemical(name).rho / 1000  # Convert from kg/m^3 to g/cm^3
    except Exception as e:
        print(f"Density lookup failed: {e}")
        density = None
        
    return {
        'Molar_Mass_g_mol': molar_mass,
        'Charge': charge,
        'Density_g_cm3': density
    }

# Example usage
smiles = 'CCO'  # ethanol
properties = get_properties_from_smiles(smiles)

for key, value in properties.items():
    print(f"{key}: {value}")


In [ ]:
found_rows

In [ ]:
Charge=[]
Molar_Mass_g_mol=[]
Density_g_cm3=[]

for _, row in tqdm(found_rows.iterrows(), total=len(found_rows), desc="Calculating properties"):
    properties=get_properties_from_smiles(row.can_SMILES)
    Charge.append(properties['Charge'])
    Molar_Mass_g_mol.append(properties['Molar_Mass_g_mol'])
    Density_g_cm3.append(properties['Density_g_cm3'])
    time.sleep(0.2) # Sleep for 0.2 seconds to limit to 5 queries per second

found_rows['Charge']=Charge
found_rows['Molar_Mass_g_mol']=Molar_Mass_g_mol
found_rows['Density_g_cm3']=Density_g_cm3
found_rows

In [ ]:
found_rows

In [ ]:
found_rows['Density_g_cm3'] = found_rows['Density_g_cm3'].fillna(1)
found_rows

In [ ]:
found_rows = found_rows[found_rows['Charge'] == 0]
found_rows = found_rows[~found_rows['can_SMILES'].str.contains('.', regex=False)]
found_rows

In [ ]:
found_rows['MD_ID'] = range(1, len(found_rows) + 1)
found_rows

In [ ]:
found_rows[['Compound_Name', 'can_SMILES', 'source', 'Charge', 'Molar_Mass_g_mol', 'Density_g_cm3', 'MD_ID']].to_csv(f'../data/processed/SelfDiff_MDmolecules.csv', index=False)

# From SelfDiff_exp

In [ ]:
filename='SelfDiff_exp'
df = pd.read_csv(f'../data/processed/{filename}_processed.csv')
df

In [ ]:
# df = df[(df.T_K >= 295) & (df.T_K <= 300)]
# df

In [ ]:
found_rows = df[df['SMILES_PubChem'] != 'Not found']
found_rows

In [ ]:
found_rows.drop_duplicates(subset='can_SMILES', inplace=True)
found_rows.reset_index(drop=True, inplace=True)
found_rows

In [ ]:
found_rows['source'] = filename
found_rows

In [ ]:
def get_properties_from_smiles(smiles):
    # Parse SMILES with RDKit
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception as e:
        print(f"Error parsing SMILES '{smiles}': {e}")
        mol = None
    
    # 1. Molar Mass
    try:
        molar_mass = Chem.Descriptors.MolWt(mol)
    except Exception as e:
        print(f"Error calculating molar mass for '{smiles}': {e}")
        molar_mass = None
    
    # 2. Net Charge
    try:
        charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    except Exception as e:
        print(f"Charge calculation failed for '{smiles}': {e}")
        charge = None
    
    # 2.5. Name from SMILES using PubChemPy
    try:
        compounds = pcp.get_compounds(smiles, 'smiles')
        name = compounds[0].iupac_name if compounds else None
    except Exception as e:
        print(f"PubChem name lookup failed: {e}")
        name = None
    
    # 3. Density (via Chemical)
    try:
        density = Chemical(name).rho / 1000  # Convert from kg/m^3 to g/cm^3
    except Exception as e:
        print(f"Density lookup failed: {e}")
        density = None
        
    return {
        'Molar_Mass_g_mol': molar_mass,
        'Charge': charge,
        'Density_g_cm3': density
    }

# Example usage
smiles = 'CCO'  # ethanol
properties = get_properties_from_smiles(smiles)

for key, value in properties.items():
    print(f"{key}: {value}")


In [ ]:
Charge=[]
Molar_Mass_g_mol=[]
Density_g_cm3=[]

for _, row in tqdm(found_rows.iterrows(), total=len(found_rows), desc="Calculating properties"):
    properties=get_properties_from_smiles(row.can_SMILES)
    Charge.append(properties['Charge'])
    Molar_Mass_g_mol.append(properties['Molar_Mass_g_mol'])
    Density_g_cm3.append(properties['Density_g_cm3'])
    time.sleep(0.2) # Sleep for 0.2 seconds to limit to 5 queries per second

found_rows['Charge']=Charge
found_rows['Molar_Mass_g_mol']=Molar_Mass_g_mol
found_rows['Density_g_cm3']=Density_g_cm3
found_rows

In [ ]:
found_rows['Density_g_cm3'] = found_rows['Density_g_cm3'].fillna(1)
found_rows

In [ ]:
found_rows = found_rows[found_rows['Charge'] == 0]
found_rows = found_rows[~found_rows['can_SMILES'].str.contains('.', regex=False)]
found_rows

In [ ]:
initial_output = pd.read_csv('../data/processed/SelfDiff_MDmolecules.csv')
initial_output = initial_output[['Compound_Name', 'can_SMILES', 'source', 'Charge', 'Molar_Mass_g_mol', 'Density_g_cm3', 'MD_ID']]
initial_output

In [ ]:
initial_output.columns

In [ ]:
found_rows.columns

In [ ]:
# Find can_SMILES in df that are not in initial_ouput
new_can_SMILES = set(found_rows['can_SMILES']) - set(initial_output['can_SMILES'])
df_new = found_rows[found_rows['can_SMILES'].isin(new_can_SMILES)].copy()

# Add missing columns to df_new with default values if needed
for col in initial_output.columns:
    if col not in df_new.columns:
        if col == 'MD_ID':
            df_new[col] = None
        elif col == 'systemprep':
            df_new[col] = None

# Reorder columns to match initial_output
df_new = df_new[initial_output.columns]

# Concatenate and reset index
updated_output = pd.concat([initial_output, df_new], ignore_index=True)

# For overlapping can_SMILES, combine source values from both dataframes
overlap_can_SMILES = set(found_rows['can_SMILES']) & set(initial_output['can_SMILES'])

# Create a mapping from can_SMILES to source in initial_output
initial_source_map = initial_output.set_index('can_SMILES')['source'].to_dict()

# Update the 'source' column in updated_output for overlaps
def combine_sources(row):
    if row['can_SMILES'] in overlap_can_SMILES:
        sources = set()
        # Get all 'source' values from df for this can_SMILES
        sources.update(found_rows[found_rows['can_SMILES'] == row['can_SMILES']]['source'].dropna().astype(str).tolist())
        # Get all 'source' values from initial_output for this can_SMILES
        sources.update(initial_output[initial_output['can_SMILES'] == row['can_SMILES']]['source'].dropna().astype(str).tolist())
        return ';'.join(sorted(sources))
    else:
        return row['source']

updated_output['source'] = updated_output.apply(combine_sources, axis=1)
updated_output

In [ ]:
updated_output

In [ ]:
# Find the maximum existing ID (ignoring None), then assign incrementing IDs to rows with None
existing_ids = updated_output['MD_ID'].dropna()
if not existing_ids.empty:
    # Convert to int if not already
    max_id = int(existing_ids.astype(int).max())
else:
    max_id = -1

# Get boolean mask for rows with None in 'ID'
none_id_mask = updated_output['MD_ID'].isna()

# Assign new IDs starting from max_id + 1
new_ids = range(max_id + 1, max_id + 1 + none_id_mask.sum())
updated_output.loc[none_id_mask, 'MD_ID'] = list(new_ids)

# If you want IDs as int type
updated_output['MD_ID'] = updated_output['MD_ID'].astype(int)

updated_output

In [ ]:
updated_output.to_csv('../data/processed/SelfDiff_MDmolecules.csv', index=False)

# From ChEMBL v34 (batch 1)

In [ ]:
filename='chembl_batch1'
df = pd.read_csv(f'../data/processed/{filename}_processed.csv')
df

In [ ]:
df.dropna(subset='can_SMILES', inplace=True)
df.drop_duplicates(subset='can_SMILES', inplace=True)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
df['source'] = filename
df

In [ ]:
Charge=[]
Molar_Mass_g_mol=[]
Density_g_cm3=[]

for _, row in tqdm(df.iterrows(), total=len(df), desc="Calculating properties"):
    properties=get_properties_from_smiles(row.can_SMILES)
    Charge.append(properties['Charge'])
    Molar_Mass_g_mol.append(properties['Molar_Mass_g_mol'])
    Density_g_cm3.append(properties['Density_g_cm3'])
    time.sleep(0.2) # Sleep for 0.2 seconds to limit to 5 queries per second

df['Charge']=Charge
df['Molar_Mass_g_mol']=Molar_Mass_g_mol
df['Density_g_cm3']=Density_g_cm3
df

In [ ]:
df

In [ ]:
df['Density_g_cm3'] = df['Density_g_cm3'].fillna(1)
df = df[df['Charge'] == 0]
df = df[~df['can_SMILES'].str.contains('.', regex=False)]
df


In [ ]:
initial_output = pd.read_csv('../data/processed/SelfDiff_MDmolecules.csv')
# initial_output = initial_output[['Compound_Name', 'can_SMILES', 'source', 'Charge', 'Molar_Mass_g_mol', 'Density_g_cm3', 'MD_ID']]
initial_output

In [ ]:
# Find can_SMILES in df that are not in initial_ouput
new_can_SMILES = set(df['can_SMILES']) - set(initial_output['can_SMILES'])
df_new = df[df['can_SMILES'].isin(new_can_SMILES)].copy()

# Add missing columns to df_new with default values if needed
for col in initial_output.columns:
    if col not in df_new.columns:
        if col == 'MD_ID':
            df_new[col] = None
        elif col == 'systemprep':
            df_new[col] = None
        elif col == 'Compound_Name':
            df_new[col] = None

# Reorder columns to match initial_output
df_new = df_new[initial_output.columns]

# Concatenate and reset index
updated_output = pd.concat([initial_output, df_new], ignore_index=True)

# For overlapping can_SMILES, combine source values from both dataframes
overlap_can_SMILES = set(df['can_SMILES']) & set(initial_output['can_SMILES'])

# Create a mapping from can_SMILES to source in initial_output
initial_source_map = initial_output.set_index('can_SMILES')['source'].to_dict()

# Update the 'source' column in updated_output for overlaps
def combine_sources(row):
    if row['can_SMILES'] in overlap_can_SMILES:
        sources = set()
        # Get all 'source' values from df for this can_SMILES
        sources.update(df[df['can_SMILES'] == row['can_SMILES']]['source'].dropna().astype(str).tolist())
        # Get all 'source' values from initial_output for this can_SMILES
        sources.update(initial_output[initial_output['can_SMILES'] == row['can_SMILES']]['source'].dropna().astype(str).tolist())
        return ';'.join(sorted(sources))
    else:
        return row['source']

updated_output['source'] = updated_output.apply(combine_sources, axis=1)
updated_output

In [ ]:
# Find the maximum existing ID (ignoring None), then assign incrementing IDs to rows with None
existing_ids = updated_output['MD_ID'].dropna()
if not existing_ids.empty:
    # Convert to int if not already
    max_id = int(existing_ids.astype(int).max())
else:
    max_id = -1

# Get boolean mask for rows with None in 'ID'
none_id_mask = updated_output['MD_ID'].isna()

# Assign new IDs starting from max_id + 1
new_ids = range(max_id + 1, max_id + 1 + none_id_mask.sum())
updated_output.loc[none_id_mask, 'MD_ID'] = list(new_ids)

# If you want IDs as int type
updated_output['MD_ID'] = updated_output['MD_ID'].astype(int)

updated_output

In [ ]:
updated_output.to_csv('../data/processed/SelfDiff_MDmolecules.csv', index=False)